# ATTENTION MECHANISM

### >Simple self-attention mechanism without trainable weights

In [1]:
import torch

# Format tensors to show 4 decimal places, avoid scientific notation, and look cleaner
torch.set_printoptions(precision=4, sci_mode=False, edgeitems=5)


inputs = torch.tensor(
    [[0.43, 0.15, 0.89],  # Your      (x^1)
     [0.55, 0.87, 0.66],  # journey   (x^2)
     [0.57, 0.85, 0.64],  # starts    (x^3)
     [0.22, 0.58, 0.33],  # with      (x^4)
     [0.77, 0.25, 0.10],  # one       (x^5)
     [0.05, 0.80, 0.55]]  # step      (x^6)
)

In [2]:
input_query=inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [3]:
input1=inputs[0]
print(f"dot product of inputs {input_query} and {input1} is:-")
print(torch.dot(input1,input_query))

dot product of inputs tensor([0.5500, 0.8700, 0.6600]) and tensor([0.4300, 0.1500, 0.8900]) is:-
tensor(0.9544)


In [4]:
query=inputs[1] #2nd input token 
attention_score=torch.empty(inputs.shape[0])
for i,x in enumerate(inputs):
    attention_score[i]=torch.dot(x,query) #score of every token based on 2nd input token 
print(attention_score) 

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


## Normalization using softmax 

In [5]:
temp_weights=attention_score / attention_score.sum()
temp_weights

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [6]:
temp_weights.sum()

tensor(1.0000)

In [7]:
def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
softmax(attention_score)    

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [8]:
attention_weights=torch.softmax(attention_score,dim=0)

In [9]:
query = inputs[1]  # 2nd input token is the query

context_vector = torch.zeros(query.shape)

for i, x in enumerate(inputs):
    print(f"{attention_weights[i]}---------->{inputs[i]}")
    context_vector += attention_weights[i] * x

print(context_vector) #for input token 2

0.13854756951332092---------->tensor([0.4300, 0.1500, 0.8900])
0.2378913015127182---------->tensor([0.5500, 0.8700, 0.6600])
0.23327402770519257---------->tensor([0.5700, 0.8500, 0.6400])
0.12399158626794815---------->tensor([0.2200, 0.5800, 0.3300])
0.10818186402320862---------->tensor([0.7700, 0.2500, 0.1000])
0.15811361372470856---------->tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


### for all tokens

In [10]:
attn_score=torch.empty(6,6)
for i,x in enumerate(inputs):
    for j,y in enumerate(inputs):
        attn_score[i,j]=torch.dot(x,y) #score of every token based on 2nd input token 
print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [11]:
attn_score=inputs @ inputs.T
print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [12]:
attn_weights=torch.softmax(attn_score,dim=1)
print(attn_weights)
print(attn_weights.sum(dim=1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [13]:
context_vec=attn_weights @ inputs
context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# SELF-ATTENTION WITH TRAINABLE WEIGHTS

In [14]:
x2=inputs[1]
din=inputs.shape[1] #input feature dimension (6, 3)
dout=2 #output (embedding) dimension

In [15]:
torch.manual_seed(123)
w_query=torch.nn.Parameter(torch.rand(din,dout)) #"This tensor is a trainable parameter" 
w_key=torch.nn.Parameter(torch.rand(din,dout)) #torch.rand(din, dout) Creates a matrix of random numbers between 0 and 1
w_value=torch.nn.Parameter(torch.rand(din,dout))

In [16]:
query_2= x2 @ w_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [18]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [20]:
w_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)

In [23]:
keys= inputs @ w_key
value= inputs @ w_value
print(f"{keys}")

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)


In [24]:
keys_2 = keys[1]
attn_score_22=torch.dot(query_2,keys_2) 

In [25]:
attn_score_22 #This is called attention score (2,2)

tensor(1.8524, grad_fn=<DotBackward0>)

In [26]:
attn_score_2=query_2 @ keys.T
attn_score_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [27]:
d_k=keys.shape[1] #2
attn_weights_2=torch.softmax(attn_score_2/d_k**0.5,dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [29]:
sum(attn_weights_2)

tensor(1., grad_fn=<AddBackward0>)

In [31]:
context_vec_2=attn_weights_2 @ value
context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

# CONTEXT VECTOR FOR ALL INPUTS 

In [35]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, din , dout):
        super().__init__()
        self.w_query=torch.nn.Parameter(torch.rand(din,dout)) #"This tensor is a trainable parameter" 
        self.w_key=torch.nn.Parameter(torch.rand(din,dout)) #torch.rand(din, dout) Creates a matrix of random numbers between 0 and 1
        self.w_value=torch.nn.Parameter(torch.rand(din,dout))

    def forward(self,x):
        queries= inputs @ w_query
        keys= inputs @ w_key
        value= inputs @ w_value

        attn_score=queries @ keys.T
        attn_weights_2=torch.softmax(attn_score/d_k**0.5,dim=-1)
        context_vec=attn_weights @ value
        return context_vec

torch.manual_seed(123)
sa_v1=SelfAttention_v1(din,dout)
sa_v1(inputs)
        

tensor([[0.2897, 0.8043],
        [0.3069, 0.8188],
        [0.3063, 0.8173],
        [0.2972, 0.7936],
        [0.2848, 0.7650],
        [0.3043, 0.8105]], grad_fn=<MmBackward0>)

In [ ]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(inputs)
        keys = self.W_key(inputs)
        values = self.W_value(inputs)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec


torch.manual_seed(789)

sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2(inputs)